# Stacking Regression - California Housing Dataset


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load Dataset

In [ ]:
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

df = X.copy()
df['MedHouseVal'] = y
print(df.shape)
df.head()

## EDA

In [ ]:
print(df.info())
print(df.describe())

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(y, kde=True, color='#1b5e20')
plt.title('House Value Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='Greens', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train size:', X_train.shape)
print('Test size:', X_test.shape)

## Define Base Learners and Meta Learner

In [ ]:
base_learners = [
    ('Decision Tree', DecisionTreeRegressor(max_depth=3, random_state=42)),
    ('KNN', KNeighborsRegressor(n_neighbors=5)),
    ('Random Forest', RandomForestRegressor(n_estimators=50, random_state=42)),
    ('Gradient Boosting', GradientBoostingRegressor(n_estimators=50, random_state=42))
]

meta_learner = Ridge(alpha=1.0)
print('Base Learners:', [name for name, _ in base_learners])
print('Meta Learner: Ridge Regression')

## Train Individual Models and Compare

In [ ]:
results = []
for name, reg in base_learners:
    reg.fit(X_train, y_train)
    r2 = r2_score(y_test, reg.predict(X_test))
    results.append({'Model': name, 'R2 Score': r2})
    print(f'{name}: {r2:.4f}')

## Train Stacking Model

In [ ]:
stacking_model = StackingRegressor(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5
)
stacking_model.fit(X_train, y_train)
y_pred = stacking_model.predict(X_test)

print('MAE:', mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print('R2 Score:', r2_score(y_test, y_pred))

## Performance Comparison

In [ ]:
results.append({'Model': 'Stacking', 'R2 Score': r2_score(y_test, y_pred)})
results_df = pd.DataFrame(results).sort_values('R2 Score', ascending=False)

plt.figure(figsize=(8, 4))
colors = ['#1b5e20' if m == 'Stacking' else '#66bb6a' for m in results_df['Model']]
plt.bar(results_df['Model'], results_df['R2 Score'], color=colors)
plt.ylabel('R² Score')
plt.title('Model Comparison')
plt.show()
print(results_df)

## Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test[:200], y_pred[:200], alpha=0.5, color='#1b5e20')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted')
plt.show()

## Cross Validation

In [ ]:
cv_scores = cross_val_score(stacking_model, X, y, cv=3, scoring='r2')
print('CV R2 Scores:', cv_scores)
print('Mean CV R2 Score:', cv_scores.mean())

plt.figure(figsize=(8, 4))
plt.bar(range(1, len(cv_scores)+1), cv_scores, color='#1b5e20')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('R² Score')
plt.title('Cross Validation R² Scores')
plt.legend()
plt.show()

## Actual vs Predicted Table

In [ ]:
comparison = pd.DataFrame({
    'Actual': y_test.values[:20],
    'Predicted': y_pred[:20]
})
comparison